In [46]:
import sys

import numpy as np
sys.path.append("..")

from src import nn
from src import losses

In [ ]:
# instancia de um modelo de 2 camadas ocultas
model = nn.MLP(
    input_size=2,
    hidden_sizes=[4, 2],
    output_size=1,
    hidden_activation="relu",
    output_activation="sigmoid",
    init="he",
    seed=7,
)

In [48]:
 # [x1, x2]
X_batch = np.array([
    [0.5, -1.0],   
    [2.0,  0.3],
])


y_batch = np.array([
    [1.0],
    [0.0],
])

print("X_batch:", X_batch.shape, " y_batch:", y_batch.shape)

X_batch: (2, 2)  y_batch: (2, 1)


In [49]:
for p in model.parameters():
    p.zero_grad()

prediction = model.forward(X_batch)
loss = losses.bce(prediction, y_batch)
loss.backward()

In [50]:
print("loss = %.6f" % float(loss.data))
for i, layer in enumerate(model.layers):
    print("layer %d: dL/dW = %.6f   dL/db = %.6f"
          % (i, np.linalg.norm(layer.W.grad), np.linalg.norm(layer.b.grad)))

    print("layer %d: W = %.6f   b = %.6f"
          % (i, np.linalg.norm(layer.W.data), np.linalg.norm(layer.b.data))) 

loss = 0.749111
layer 0: dL/dW = 0.353810   dL/db = 0.200521
layer 0: W = 1.986816   b = 0.000000
layer 1: dL/dW = 0.424696   dL/db = 0.093842
layer 1: W = 1.085301   b = 0.000000
layer 2: dL/dW = 0.046783   dL/db = 0.066087
layer 2: W = 1.419974   b = 0.000000


In [51]:
from src.optim import SGD

optimizer = SGD(model.parameters(), lr=0.5)

W_t1 = [layer.W.data.copy() for layer in model.layers]
b_t1 = [layer.b.data.copy() for layer in model.layers]

# um passo de otimização com os gradientes calculados acima: novo = antigo - lr * grad
optimizer.step()

In [52]:
new_loss = losses.bce(model.forward(X_batch), y_batch)
print("loss antes = %.6f loss depois = %.6f" % (float(loss.data), float(new_loss.data)))

for i, layer in enumerate(model.layers):
    print("\nlayer %d" % i)
    print("W antes:\n", W_t1[i])
    print("W depois:\n", layer.W.data)
    print("b antes: ", b_t1[i])
    print("b depois:", layer.b.data)

loss antes = 0.749111 loss depois = 0.678486

layer 0
W antes:
 [[ 1.23015336e-03  2.98745538e-01 -2.74137855e-01 -8.90591839e-01]
 [-4.54670785e-01 -9.91646555e-01  6.01436026e-02  1.34021525e+00]]
W depois:
 [[ 0.05043248  0.38862498 -0.27413786 -0.89059184]
 [-0.55307545 -0.8862297   0.0601436   1.34021525]]
b antes:  [[0. 0. 0. 0.]]
b depois: [[ 0.09840466 -0.01920093  0.          0.        ]]

layer 1
W antes:
 [[-0.34804257 -0.43874201]
 [ 0.34637064  0.25235722]
 [ 0.07453913 -0.65794026]
 [-0.02068416  0.4916536 ]]
W depois:
 [[-0.43811453 -0.46940553]
 [ 0.16666095  0.19117801]
 [ 0.07453913 -0.65794026]
 [-0.02068416  0.4916536 ]]
b antes:  [[0. 0.]]
b depois: [[-0.04441766 -0.01512126]]

layer 2
W antes:
 [[-1.34421455]
 [-0.45761576]]
W depois:
 [[-1.32122917]
 [-0.45327666]]
b antes:  [[0.]]
b depois: [[0.03304358]]
